# Sonus — Google Colab Quickstart

Run the Sonus multi-engine TTS server in Colab and access it from anywhere.

**MLX engines** (Qwen3, Chatterbox) require Apple Silicon and are skipped on Colab.
**ONNX engines** (Kokoro, Piper) work cross-platform and are available below.

## 1. Select engine

In [ ]:
# @title Engine selection
ENGINE = "kokoro"  # @param ["kokoro", "piper", "both"]
PIPER_VOICE = "en_US-lessac-medium"  # @param {type:"string"}
print(f"Selected engine: {ENGINE}")
if ENGINE != "kokoro":
    print(f"Piper voice: {PIPER_VOICE}")

## 2. Install dependencies

In [ ]:
import os, sys, subprocess, time, urllib.request, json

REPO_DIR = "/content/sonus"
os.chdir("/content")

# Clone repo if not already cloned
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/guptasanchit90/sonus.git
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)

print("\n[1/3] Installing Python packages...")
!pip install -q -r requirements-colab.txt

print("\n[2/3] Installing ffmpeg...")
!apt-get -qq install -y ffmpeg

print("\n[3/3] Verifying install...")
import kokoro_onnx
import piper
import fastapi
import soundfile
print("All dependencies ready.")

## 3. Download models

In [ ]:
print(f"Downloading models for: {ENGINE}")
!bash scripts/download_models.sh models {ENGINE} {PIPER_VOICE} 2>&1
print("\nModel files:")
!find models -type f -not -path '*/.*' | sort

## 4. Start the server

In [ ]:
# Kill any previous instance
!pkill -f "uvicorn server:app" || true
!pkill -f "python server.py" || true
time.sleep(1)

# Start server in background
LOG_FILE = "/content/sonus_server.log"
!nohup python server.py > {LOG_FILE} 2>&1 &
time.sleep(3)

# Check startup
with open(LOG_FILE) as f:
    log = f.read()
print(log[-2000:] if len(log) > 2000 else log)

if "Uvicorn running on" in log:
    print("\nServer is running!")
else:
    print("\nWaiting for server...")
    time.sleep(5)
    with open(LOG_FILE) as f:
        print(f.read()[-2000:])

## 5. Expose via public URL

Choose one method to make the server accessible from outside Colab.

In [ ]:
# Expose via localtunnel
!npm install -g localtunnel 2>/dev/null
!nohup npx localtunnel --port 8000 > /content/tunnel.log 2>&1 &
time.sleep(3)
with open("/content/tunnel.log") as f:
    content = f.read()
import re
urls = re.findall(r'https?://[^\s]+', content)
if urls:
    base = urls[0].rstrip("/")
    print(f"API:         {base}/v1")
    print(f"Web UI:      {base}")
    print(f"Docs:        {base}/api-docs")
else:
    print("Localtunnel starting... check logs:")
    print(content)

## 6. Test it out

In [ ]:
BASE = "http://localhost:8000"

# Health check
req = urllib.request.Request(f"{BASE}/health")
resp = urllib.request.urlopen(req)
print("Health:", resp.read().decode())

# List models
req = urllib.request.Request(f"{BASE}/v1/models")
resp = urllib.request.urlopen(req)
models = json.loads(resp.read().decode())
print(f"\nAvailable models ({len(models['data'])}):")
for m in models['data']:
    print(f"  {m['id']}  ({m.get('engine', '?')})")

# Generate a quick test
payload = json.dumps({
    "model": "tts-1",
    "input": "Hello from Sonus running in Google Colab!",
    "voice": "af_heart",
    "response_format": "wav"
}).encode()
req = urllib.request.Request(
    f"{BASE}/v1/audio/speech",
    data=payload,
    headers={"Content-Type": "application/json"}
)
resp = urllib.request.urlopen(req)
with open("/content/test_output.wav", "wb") as f:
    f.write(resp.read())
print(f"\nTest audio saved to /content/test_output.wav")
print(f"Seed: {resp.headers.get('X-Seed', 'N/A')}")
print(f"Duration: {resp.headers.get('X-Audio-Duration', 'N/A')}s")

## 7. Use from anywhere

Point any OpenAI-compatible client to your tunnel URL:

```python
from openai import OpenAI
client = OpenAI(base_url="<YOUR_TUNNEL_URL>/v1", api_key="not-needed")

response = client.audio.speech.create(
    model="tts-1",
    voice="af_heart",
    input="Sonus is running in the cloud!"
)
response.stream_to_file("output.mp3")
```

Or use curl:

```
curl <YOUR_TUNNEL_URL>/v1/audio/speech \
  -H "Content-Type: application/json" \
  -d '{"model": "tts-1", "voice": "af_heart", "input": "Hello from Sonus!"}' \
  --output speech.mp3
```